In [7]:
from dotenv import load_dotenv
load_dotenv("./../.env")
import os
langsmith_api = os.environ["LANGSMITH_API_KEY"]
if langsmith_api:
    print("LangSmith API key loaded successfully.")

LangSmith API key loaded successfully.


In [8]:
from langchain_ollama import ChatOllama
model = "llama3.2"
base_url = "http://localhost:11434"
ollama = ChatOllama(model=model, base_url=base_url, temperature=0.9, num_predict=256)

In [9]:
# 问题分类模块
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
cls_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a question classifier, you classify the user's question into below classes: \n"
               "1. technology\n"
               "2. art\n"
               "Please classify the human question into one of the above classes, and only return the class name."),
    ("human", "{question}")
])

question_cls_chain = cls_prompt | ollama | StrOutputParser()

In [10]:
# 回答科技问题模块
tech_prompt = ChatPromptTemplate.from_template("""
                                               You are a technology expert.
                                               You answer question in a concise and informative way.
                                               Please answer the following question in {number} points:
                                               Question: {question}
                                               Answer:"""
                                               )
tech_chain = tech_prompt | ollama | StrOutputParser()

In [11]:
# 回答艺术问题模块
art_prompt = ChatPromptTemplate.from_template("""
                                               You are an art expert.
                                               You answer question in a casual and abstract way.
                                               Please answer the following question in {number} points:
                                               Question: {question}
                                               Answer:"""
                                               )
art_chain = art_prompt | ollama | StrOutputParser()

In [12]:
# 问题分类处理模块
from langchain_core.runnables import chain
@chain
def handle_question(input):
    if "technology" in input["cls"]:
        return tech_chain.invoke({"question": input["question"], "number": input["number"]})
    elif "art" in input["cls"]:
        return art_chain.invoke({"question": input["question"], "number": input["number"]})
    else:
        return StrOutputParser().parse("Error")

In [27]:
# 组装成一个完整的chain
#question = "What kind of art style is the Mona Lisa painting? Can you give me 3 points to explain it?"
question = "What is the difference between AI and Machine Learning? Can you give me 3 points to explain it?"
number = 3

full_chain = ({"cls":question_cls_chain, "question": lambda x:x["question"], "number":lambda x:x["number"]} 
              | handle_question)
result = full_chain.invoke({"question": question, "number": number})
print(result)

Here are 3 key differences between Artificial Intelligence (AI) and Machine Learning (ML):

1. **Purpose**: AI is a broader field that encompasses various techniques, including ML, to create intelligent machines that can perform tasks that typically require human intelligence. ML is a subset of AI focused specifically on developing algorithms and statistical models that enable machines to learn from data.

2. **Learning Process**: In AI, learning occurs through rule-based systems or logical deduction. In contrast, ML involves training algorithms using large datasets to improve their performance on specific tasks, such as image classification or natural language processing.

3. **Deployment**: AI is often deployed in applications where the system needs to make decisions based on rules and logical reasoning. ML, on the other hand, is typically used in applications where data-driven decision-making is required, such as predictive analytics, recommendation systems, and autonomous vehicles.

In [2]:
# 使用parser将结果解析成结构化数据
# 创建输出数据格式模型：
from pydantic import BaseModel, Field
class Answer(BaseModel):
    main:str = Field(description="The main part of the answer, include the key points.")
    question_cls:str = Field(description="The question class, technology or art.")
    points:int = Field(description="The number of points in the answer.")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser, CommaSeparatedListOutputParser


parser_pydantic = PydanticOutputParser(pydantic_object=Answer)
instruction_pydantic = parser_pydantic.get_format_instructions()

parser_json = JsonOutputParser(pydantic_object=Answer)
instruction_json = parser_json.get_format_instructions()

parser_list = CommaSeparatedListOutputParser()
instruction_list = parser_list.get_format_instructions()


In [15]:
# 组装Prompt
parser_prompt = ChatPromptTemplate.from_template("""
                                                   You are a question classifier, 
                                                   You answer user's question in {number} points,
                                                   Here is your formatting instruction: {instruction}
                                                    Question: {question}
                                                    Answer:
                                                   """
                                                   )
print(parser_prompt)

input_variables=['instruction', 'number', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['instruction', 'number', 'question'], input_types={}, partial_variables={}, template="\n                                                   You are a question classifier, \n                                                   You answer user's question in {number} points,\n                                                   Here is your formatting instruction: {instruction}\n                                                    Question: {question}\n                                                    Answer:\n                                                   "), additional_kwargs={})]


In [32]:
question = "Tell me about lions"
number = 1
instruction = instruction_json
parser_chain = parser_prompt | ollama | parser_json
parser_chain.invoke({"question": question, "number": number, "instruction": instruction})

{'properties': {'main': 'Lions are large carnivorous mammals that belong to the Felidae family.',
  'question_cls': 'Animal Kingdom',
  'points': '3'}}

In [34]:
# 当前更稳妥的方案，但需要模型有tool功能
structured_ollama = ollama.with_structured_output(Answer)
answer = structured_ollama.invoke(question)
print(answer)

main="The Lion's Roar: Unveiling the Majestic Lion" question_cls='' points=5
